In [1]:
import sys
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Python: 3.10.19 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 16:41:31) [MSC v.1929 64 bit (AMD64)]
Torch: 2.5.1
CUDA available: False


In [2]:
import whisper
from transformers import pipeline

print("Loading Whisper model...")
whisper_model = whisper.load_model("small")

print("Loading BART summarizer...")
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

print("Models loaded successfully")


Loading Whisper model...
Loading BART summarizer...


Device set to use cpu


Models loaded successfully


In [3]:
audio_path = "Sample_audio.mpeg" 

result = whisper_model.transcribe(
    audio_path,
    fp16=False
)

transcript = result["text"]

print("Transcription completed")
print("Total words:", len(transcript.split()))


Transcription completed
Total words: 766


In [4]:
print("===== TRANSCRIPT PREVIEW =====")
print(transcript[:600])


===== TRANSCRIPT PREVIEW =====
 I'll start again. Thank you very much. This is the arm of Springfield meeting agenda for April 8th of 2025 committee of a whole starting at exactly 1pm. I'd like to do the introductions myself. I'm Mayor Patrick Tarion into my right and descending order is Deputy Mayor Glenn fuel Ward 1 Ward 2 is Andy Kaczynski. The next will be Ward 3 Mark Miller. Next would be Ward 4. Excuse me, Melinda Warren. And I wonder if we'll go to the proof of the agenda. Is there any amendments or deletions to that from Council? I see none. Then I get a show hands of those in support. unanimous. It's carried. And w


In [5]:
import re

def chunk_text(text, chunk_size=600):
    words = text.split()
    return [
        " ".join(words[i:i + chunk_size])
        for i in range(0, len(words), chunk_size)
    ]


def extract_section(text, focus):
    prompt = f"{focus}:\n{text}"
    s = summarizer(
        prompt,
        max_length=120,
        min_length=40,
        do_sample=False
    )
    return s[0]["summary_text"]


def clean_section(text, section_name):
    text = text.strip()

    redundant_phrases = [
        "If I get a mover and second to go into a closed meeting",
        "And we're prepared to adjourn",
        "We discussed legal matters"
    ]
    for phrase in redundant_phrases:
        text = text.replace(phrase, "").strip()

    if section_name == "Action Items":
        keywords = ["will", "assigned", "responsible", "to prepare", "to submit"]
        if not any(k in text.lower() for k in keywords):
            return "No specific action items were identified during the meeting."

    return text


def polish_text(text):
    sentences = [s.strip() for s in text.split(".") if s.strip()]
    unique = []
    for s in sentences:
        if s not in unique:
            unique.append(s)

    text = ". ".join(unique)
    text = text.replace("..", ".").replace(" .", ".").strip()

    if not text.endswith("."):
        text += "."

    return text


def to_bullets(text, max_points=3):
    sentences = re.split(r'(?<=[.!?])\s+', text)

    bullets = []
    for s in sentences:
        s = s.strip()
        if len(s) < 20:
            continue
        if s.lower().startswith("no specific action"):
            bullets = [s]
            break
        bullets.append(s)
        if len(bullets) == max_points:
            break

    if not bullets:
        bullets = [text.strip()]

    return "\n".join([f"• {b}" for b in bullets])


In [6]:
chunks = chunk_text(transcript)

print("===== CHUNKING INFO =====")
print("Total chunks:", len(chunks))
print("\nFirst chunk preview:\n")
print(chunks[0][:500])


===== CHUNKING INFO =====
Total chunks: 2

First chunk preview:

I'll start again. Thank you very much. This is the arm of Springfield meeting agenda for April 8th of 2025 committee of a whole starting at exactly 1pm. I'd like to do the introductions myself. I'm Mayor Patrick Tarion into my right and descending order is Deputy Mayor Glenn fuel Ward 1 Ward 2 is Andy Kaczynski. The next will be Ward 3 Mark Miller. Next would be Ward 4. Excuse me, Melinda Warren. And I wonder if we'll go to the proof of the agenda. Is there any amendments or deletions to that fr


In [7]:
summaries = []

for i, chunk in enumerate(chunks):
    print(f"Summarizing chunk {i+1}/{len(chunks)}")
    s = summarizer(
        chunk,
        max_length=180,
        min_length=60,
        do_sample=False
    )
    summaries.append(s[0]["summary_text"])

meeting_summary = " ".join(summaries)


Summarizing chunk 1/2
Summarizing chunk 2/2


In [8]:
print("===== MEETING SUMMARY PREVIEW =====")
print(meeting_summary[:600])


===== MEETING SUMMARY PREVIEW =====
This is the arm of Springfield meeting agenda for April 8th of 2025 committee of a whole starting at exactly 1pm. I'd like to do the introductions myself. I'm Mayor Patrick Tarion into my right and descending order is Deputy Mayor Glenn fuel Ward 1 Ward 2 is Andy Kaczynski. The next will be Ward 3 Mark Miller. Next would be Ward 4. Excuse me, Melinda Warren. Council was satisfied with the fine amounts for this bylaw. We discussed legal matters. And we're prepared to adjourn at 138 p.m. If I get a mover and second to go into a closed meeting. Glenn and Patrick. Then we'll go into closed meeting


In [9]:
mom = {
    "Agenda": extract_section(
        meeting_summary,
        "Agenda items discussed in the meeting"
    ),
    "Key Discussion Summary": extract_section(
        meeting_summary,
        "Key discussion points from the meeting"
    ),
    "Decisions": extract_section(
        meeting_summary,
        "Decisions taken during the meeting"
    ),
    "Action Items": extract_section(
        meeting_summary,
        "Action items identified in the meeting"
    )
}


In [10]:
print("===== RAW MOM (BEFORE CLEANUP) =====")

for section, content in mom.items():
    print(f"\n{section}:")
    print(content)


===== RAW MOM (BEFORE CLEANUP) =====

Agenda:
Council was satisfied with the fine amounts for this bylaw. We discussed legal matters. And we're prepared to adjourn at 138 p.m. If I get a mover and second to go into a closed meeting.

Key Discussion Summary:
Council was satisfied with the fine amounts for this bylaw. We discussed legal matters. And we're prepared to adjourn at 138 p.m. If I get a mover and second to go into a closed meeting.

Decisions:
Mayor Patrick Tarion introduced the council. Council was satisfied with the fine amounts for this bylaw. We discussed legal matters. And we're prepared to adjourn at 138 p.m. If I get a mover and second to go into a closed meeting.

Action Items:
Mayor Patrick Tarion introduced the council. Council was satisfied with the fine amounts for this bylaw. We discussed legal matters. And we're prepared to adjourn at 138 p.m. If I get a mover and second to go into a closed meeting.


In [11]:
polished_mom = {}

for section, content in mom.items():
    cleaned = clean_section(content, section)
    polished = polish_text(cleaned)
    polished_mom[section] = polished


In [12]:
bullet_mom = {}

for section, content in polished_mom.items():
    bullet_mom[section] = to_bullets(content, max_points=3)


In [13]:
meeting_title = input("Enter Meeting Title: ").strip()
meeting_date = input("Enter Meeting Date (e.g. 8 April 2025): ").strip()
meeting_time = input("Enter Meeting Time (e.g. 1:00 PM – 1:38 PM): ").strip()

if not meeting_title:
    meeting_title = "Meeting"
if not meeting_date:
    meeting_date = "Not specified"
if not meeting_time:
    meeting_time = "Not specified"


Enter Meeting Title:  Springfield meeting
Enter Meeting Date (e.g. 8 April 2025):  8 April 2025
Enter Meeting Time (e.g. 1:00 PM – 1:38 PM):  12:00 AM


In [14]:
def format_business_mom(title, date, time, mom_sections):
    return f"""
MINUTES OF MEETING

Meeting Title: {title}
Date: {date}
Time: {time}

--------------------------------------------------

Agenda:
{mom_sections['Agenda']}

Key Discussion Summary:
{mom_sections['Key Discussion Summary']}

Decisions:
{mom_sections['Decisions']}

Action Items:
{mom_sections['Action Items']}

--------------------------------------------------
"""


In [15]:
final_document = format_business_mom(
    meeting_title,
    meeting_date,
    meeting_time,
    bullet_mom
)

print(final_document)



MINUTES OF MEETING

Meeting Title: Springfield meeting
Date: 8 April 2025
Time: 12:00 AM

--------------------------------------------------

Agenda:
• Council was satisfied with the fine amounts for this bylaw.

Key Discussion Summary:
• Council was satisfied with the fine amounts for this bylaw.

Decisions:
• Mayor Patrick Tarion introduced the council.
• Council was satisfied with the fine amounts for this bylaw.

Action Items:
• No specific action items were identified during the meeting.

--------------------------------------------------

